In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/UNetRes/2026/TWO_quintile_clim-20260831-mslp.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
#print(ds.variables)
# 如果需要查看数据集的维度，可以使用
#print(ds.dims)


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_u_20260831.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
print(ds.variables)
# 如果需要查看数据集的维度，可以使用
print(ds.dims)


In [ ]:
# 1 UNetq1
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_u_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/UNetRes/2026/TWO_quintile_clim-20260831-mslp.nc"

output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测"
os.makedirs(output_dir, exist_ok=True)

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-UNet-20260831-mslp-q1.nc"


# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 使用新的气候四分位数 q1（去掉 time）
mslp_clim = clim_ds["q1"].isel(time=0)

# 第一周四天
first_week_vars = [
    "mslp_mon",
    "mslp_wed",
    "mslp_fri",
    "mslp_sun"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位概率函数（原逻辑不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 对第一周四天分别计算，再求平均
# ===============================
daily_prob_list = []

for var in first_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

# 在 day 维度上取平均
mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob1"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第一周四天平均后的五分位概率文件已保存到：")
print(output_file)


In [ ]:
# 2 UNetq2
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_u_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/UNetRes/2026/TWO_quintile_clim-20260831-mslp.nc"

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-UNet-20260831-mslp-q2.nc"

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 👉 使用新的气候四分位数 q2（去掉 time 维）
mslp_clim = clim_ds["q2"].isel(time=0)

# 第二周三天
second_week_vars = [
    "mslp_tue_next",
    "mslp_thu_next",
    "mslp_sat_next"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位数概率函数（原逻辑保持不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)

    # 确保每个格点五分位概率和为 1
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 第二周三天分别计算 → 再取平均
# ===============================
daily_prob_list = []

for var in second_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob2"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第二周三天平均后的 Q2 五分位概率文件已保存到：")
print(output_file)


In [ ]:
# 3 SAq1
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_sa_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/SA/2026/TWO_quintile_clim-20260831-mslp.nc"

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-SA-20260831-mslp-q1.nc"

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 使用新的气候四分位数 q1（去掉 time）
mslp_clim = clim_ds["q1"].isel(time=0)

# 第一周四天
first_week_vars = [
    "mslp_mon",
    "mslp_wed",
    "mslp_fri",
    "mslp_sun"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位概率函数（原逻辑不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 对第一周四天分别计算，再求平均
# ===============================
daily_prob_list = []

for var in first_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

# 在 day 维度上取平均
mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob1"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第一周四天平均后的五分位概率文件已保存到：")
print(output_file)

In [ ]:
# 4 SAq2
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_sa_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/SA/2026/TWO_quintile_clim-20260831-mslp.nc"

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-SA-20260831-mslp-q2.nc"

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 👉 使用新的气候四分位数 q2（去掉 time 维）
mslp_clim = clim_ds["q2"].isel(time=0)

# 第二周三天
second_week_vars = [
    "mslp_tue_next",
    "mslp_thu_next",
    "mslp_sat_next"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位数概率函数（原逻辑保持不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)

    # 确保每个格点五分位概率和为 1
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 第二周三天分别计算 → 再取平均
# ===============================
daily_prob_list = []

for var in second_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob2"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第二周三天平均后的 Q2 五分位概率文件已保存到：")
print(output_file)


In [ ]:
# 5 TCq1
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_tc_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/TC/2026/TWO_quintile_clim-20260831-mslp.nc"

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-TC-20260831-mslp-q1.nc"

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 使用新的气候四分位数 q1（去掉 time）
mslp_clim = clim_ds["q1"].isel(time=0)

# 第一周四天
first_week_vars = [
    "mslp_mon",
    "mslp_wed",
    "mslp_fri",
    "mslp_sun"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位概率函数（原逻辑不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 对第一周四天分别计算，再求平均
# ===============================
daily_prob_list = []

for var in first_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

# 在 day 维度上取平均
mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob1"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第一周四天平均后的五分位概率文件已保存到：")
print(output_file)

In [ ]:
# 6 TCq2
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_tc_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/TC/2026/TWO_quintile_clim-20260831-mslp.nc"

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-TC-20260831-mslp-q2.nc"

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 👉 使用新的气候四分位数 q2（去掉 time 维）
mslp_clim = clim_ds["q2"].isel(time=0)

# 第二周三天
second_week_vars = [
    "mslp_tue_next",
    "mslp_thu_next",
    "mslp_sat_next"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位数概率函数（原逻辑保持不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)

    # 确保每个格点五分位概率和为 1
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 第二周三天分别计算 → 再取平均
# ===============================
daily_prob_list = []

for var in second_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob2"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第二周三天平均后的 Q2 五分位概率文件已保存到：")
print(output_file)

In [ ]:
# 7 upCG3q1
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_upCG3_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/upCG3/2026/TWO_quintile_clim-20260831-mslp.nc"

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-upCG3-20260831-mslp-q1.nc"

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 使用新的气候四分位数 q1（去掉 time）
mslp_clim = clim_ds["q1"].isel(time=0)

# 第一周四天
first_week_vars = [
    "mslp_mon",
    "mslp_wed",
    "mslp_fri",
    "mslp_sun"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位概率函数（原逻辑不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 对第一周四天分别计算，再求平均
# ===============================
daily_prob_list = []

for var in first_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

# 在 day 维度上取平均
mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob1"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第一周四天平均后的五分位概率文件已保存到：")
print(output_file)

In [ ]:
# 8 upCG3q2
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_upCG3_20260831.nc"

clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/MSLP/upCG3/2026/TWO_quintile_clim-20260831-mslp.nc"

output_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-upCG3-20260831-mslp-q2.nc"

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 👉 使用新的气候四分位数 q2（去掉 time 维）
mslp_clim = clim_ds["q2"].isel(time=0)

# 第二周三天
second_week_vars = [
    "mslp_tue_next",
    "mslp_thu_next",
    "mslp_sat_next"
]

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算五分位数概率函数（原逻辑保持不变）
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []

    for i in range(5):
        if i == 0:
            p = forecast < clim_quintiles[0]
        elif i == 4:
            p = forecast >= clim_quintiles[3]
        else:
            p = (forecast >= clim_quintiles[i - 1]) & (forecast < clim_quintiles[i])

        p = p.drop_vars(
            [v for v in p.coords if v not in ["latitude", "longitude"]],
            errors="ignore"
        )

        prob_list.append(p.astype(np.float32))

    prob = xr.concat(
        prob_list,
        dim="quintile",
        coords="minimal",
        compat="override"
    )

    prob = prob.assign_coords(quintile=quintiles)

    # 确保每个格点五分位概率和为 1
    prob = prob / prob.sum(dim="quintile")

    return prob

# ===============================
# 第二周三天分别计算 → 再取平均
# ===============================
daily_prob_list = []

for var in second_week_vars:
    mslp_day = forecast_ds[var].isel(time=0)
    prob_day = compute_quintile_prob(mslp_day, mslp_clim)
    daily_prob_list.append(prob_day)

mslp_quintile_prob = xr.concat(
    daily_prob_list,
    dim="day"
).mean(dim="day")

mslp_quintile_prob.name = "mslp_quintile_prob2"

# ===============================
# 保存 NetCDF（无 time 维）
# ===============================
mslp_quintile_prob.to_netcdf(output_file)

print("✅ 第二周三天平均后的 Q2 五分位概率文件已保存到：")
print(output_file)

In [ ]:
# 绘图
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-SA-20260831-mslp-q1.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-SA-20260831-mslp-q2.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 数据变量名不同
data_q1 = ds_q1['mslp_quintile_prob1']
data_q2 = ds_q2['mslp_quintile_prob2']

# 创建图形
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标
vmin, vmax = 0, 1
cmap = 'plasma'  # 更鲜明的概率色标

for i in range(5):
    # q1
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

fig.suptitle("MSLP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

fig.savefig("mslp_quintile_probabilities_20260831.png",
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-UNet-20260831-mslp-q1.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
print(ds.variables)
# 如果需要查看数据集的维度，可以使用
print(ds.dims)

In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-TC-20260831-mslp-q1.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
print(ds.variables)
# 如果需要查看数据集的维度，可以使用
print(ds.dims)

In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/S-fc-SA-20260831-mslp-q1.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
print(ds.variables)
# 如果需要查看数据集的维度，可以使用
print(ds.dims)

In [ ]:
# 第一个时刻
import os
import xarray as xr
import numpy as np

# === 文件目录与模型文件 ===
folder = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测"
model_files = [
    "S-fc-UNet-20260831-mslp-q1.nc",
    "S-fc-TC-20260831-mslp-q1.nc",
    "S-fc-SA-20260831-mslp-q1.nc"
]

# === 读取所有模型的数据并累加 ===
sum_probs = None

for file in model_files:
    path = os.path.join(folder, file)
    ds = xr.open_dataset(path)
    
    # 提取五分位概率变量: (quintile, lat, lon)
    prob = ds['mslp_quintile_prob1']  # shape: (5, 121, 240)
    
    if sum_probs is None:
        sum_probs = prob
    else:
        sum_probs += prob

# === 计算均值 ===
mean_probs = sum_probs / len(model_files)

# === 归一化：保证每个格点上5个分位数和为1 ===
mean_probs_normalized = mean_probs / mean_probs.sum(dim='quintile')

# === 保存为新文件 ===
output_path = os.path.join(folder, "20260831-mslp-q1-ensemble_mean.nc")
mean_probs_normalized.to_dataset(name="mslp_q1_ensemble_prob").to_netcdf(output_path)

print(f"已保存至: {output_path}")


In [ ]:
# 第二个时刻
import os
import xarray as xr
import numpy as np

# === 文件目录与模型文件 ===
folder = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测"
model_files = [
    "S-fc-UNet-20260831-mslp-q2.nc",
    "S-fc-TC-20260831-mslp-q2.nc",
    "S-fc-SA-20260831-mslp-q2.nc"
]

# === 读取所有模型的数据并累加 ===
sum_probs = None

for file in model_files:
    path = os.path.join(folder, file)
    ds = xr.open_dataset(path)
    
    # 提取五分位概率变量: (quintile, lat, lon)
    prob = ds['mslp_quintile_prob2']  # shape: (5, 121, 240)
    
    if sum_probs is None:
        sum_probs = prob
    else:
        sum_probs += prob

# === 计算均值 ===
mean_probs = sum_probs / len(model_files)

# === 归一化：保证每个格点上5个分位数和为1 ===
mean_probs_normalized = mean_probs / mean_probs.sum(dim='quintile')

# === 保存为新文件 ===
output_path = os.path.join(folder, "20260831-mslp-q2-ensemble_mean.nc")
mean_probs_normalized.to_dataset(name="mslp_q2_ensemble_prob").to_netcdf(output_path)

print(f"已保存至: {output_path}")


In [ ]:
# 绘图
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/20260831-mslp-q1-ensemble_mean.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/20260831-mslp-q2-ensemble_mean.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 数据变量名不同
data_q1 = ds_q1['mslp_q1_ensemble_prob']
data_q2 = ds_q2['mslp_q2_ensemble_prob']

# 创建图形
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标
vmin, vmax = 0, 1
cmap = 'plasma'  # 更鲜明的概率色标

for i in range(5):
    # q1
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

fig.suptitle("MSLP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

fig.savefig("mslp_quintile_probabilities_20260831.png",
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 中国绘图
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/20260831-mslp-q1-ensemble_mean.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/20260831-mslp-q2-ensemble_mean.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 数据变量名不同
data_q1 = ds_q1['mslp_q1_ensemble_prob']
data_q2 = ds_q2['mslp_q2_ensemble_prob']

# 创建图形
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标
vmin, vmax = 0, 1
cmap = 'plasma'  # 更鲜明的概率色标

for i in range(5):
    # q1
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([73, 135, 5, 54], crs=ccrs.PlateCarree())  # 限定范围
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([73, 135, 5, 54], crs=ccrs.PlateCarree())  # 限定范围
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

fig.suptitle("MSLP Ensemble Quintile Probabilities 20260831&20260831", fontsize=20)

fig.savefig("CHINAmslp_quintile_probabilities_20260831.png",
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 西欧和西非
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/20260831-mslp-q1-ensemble_mean.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-预测/20260831-mslp-q2-ensemble_mean.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 数据变量名不同
data_q1 = ds_q1['mslp_q1_ensemble_prob']
data_q2 = ds_q2['mslp_q2_ensemble_prob']

# 创建图形
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标
vmin, vmax = 0, 1
cmap = 'plasma'  # 更鲜明的概率色标

for i in range(5):
    # q1
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([-25, 40, -10, 72], crs=ccrs.PlateCarree())  # 限定范围
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='white')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([-25, 40, -10, 72], crs=ccrs.PlateCarree())  # 限定范围
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

fig.suptitle("MSLP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

fig.savefig("WESTmslp_quintile_probabilities_20260831.png",
            dpi=300, bbox_inches='tight')
plt.show()